# SmartVision - Step 2: Train ResNet-50 on CIFAR-100

**What this notebook does**

1. Loads CIFAR-100 (100 classes, 60,000 small images) automatically. No download needed.
2. Builds ResNet-50 starting from ImageNet weights (transfer learning) with a new 100-class head.
3. **Phase 1:** trains only the new head (fast). **Phase 2:** fine-tunes the deeper ResNet blocks (slow, gives the accuracy).
4. Evaluates on the 10,000 test images (top-1, top-5, weakest classes).
5. Saves the files your FastAPI backend needs: `resnet50_cifar100.keras`, `cifar100_classes.json`, `training_config.json`.

**Before you run:** turn on the GPU. Colab: `Runtime > Change runtime type > T4 GPU`. Kaggle: `Settings > Accelerator > GPU T4`.

**Time (estimate):** roughly 1.5 to 3 hours on a T4. Colab can disconnect, so run the Google Drive cell in section 1 to keep checkpoints safe.

**Settings that must match the backend `.env`:** `INPUT_SIZE=224` and `PREPROCESS=resnet`.

## 1. Setup

In [ ]:
import os, json, time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus if gpus else "NOT FOUND - training on CPU would take days. Enable a GPU first.")

# Mixed precision = about 2x faster on T4, same accuracy
if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")
print("Policy:", keras.mixed_precision.global_policy().name)

In [ ]:
# ---------- SETTINGS (edit here) ----------
INPUT_SIZE   = 224      # must match backend INPUT_SIZE
BATCH_SIZE   = 64       # lower to 32 if you get an out-of-memory error
EPOCHS_HEAD  = 5        # phase 1
EPOCHS_FINE  = 15       # phase 2
LR_HEAD      = 1e-3
LR_FINE      = 1e-4
VAL_SIZE     = 5000     # taken from the 50,000 training images
NUM_CLASSES  = 100

# Optional: keep results on Google Drive so a Colab disconnect does not lose them
USE_DRIVE = False
OUT_DIR = "outputs"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/smartvision_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving to:", OUT_DIR)

## 2. Data

CIFAR-100 class names are alphabetical. This exact list is saved next to the model, because the backend needs the **same order** to turn numbers into names.

In [ ]:
CLASS_NAMES = ['apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle','bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel','can','castle','caterpillar','cattle','chair','chimpanzee','clock','cloud','cockroach','couch','crab','crocodile','cup','dinosaur','dolphin','elephant','flatfish','forest','fox','girl','hamster','house','kangaroo','keyboard','lamp','lawn_mower','leopard','lion','lizard','lobster','man','maple_tree','motorcycle','mountain','mouse','mushroom','oak_tree','orange','orchid','otter','palm_tree','pear','pickup_truck','pine_tree','plain','plate','poppy','porcupine','possum','rabbit','raccoon','ray','road','rocket','rose','sea','seal','shark','shrew','skunk','skyscraper','snail','snake','spider','squirrel','streetcar','sunflower','sweet_pepper','table','tank','telephone','television','tiger','tractor','train','trout','tulip','turtle','wardrobe','whale','willow_tree','wolf','woman','worm']
assert len(CLASS_NAMES) == NUM_CLASSES

(x_full, y_full), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode="fine")
y_full, y_test = y_full.flatten(), y_test.flatten()

rng = np.random.RandomState(SEED)
idx = rng.permutation(len(x_full))
val_idx, tr_idx = idx[:VAL_SIZE], idx[VAL_SIZE:]
x_train, y_train = x_full[tr_idx], y_full[tr_idx]
x_val,   y_val   = x_full[val_idx], y_full[val_idx]
print("train:", x_train.shape, " val:", x_val.shape, " test:", x_test.shape)

# quick look
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(x_train[i]); ax.set_title(CLASS_NAMES[y_train[i]], fontsize=9); ax.axis("off")
plt.show()

## 3. Input pipeline

- **Training images** get light augmentation (random flip, random shift) so the model does not just memorise.
- Every image is enlarged to 224x224 and passed through ResNet's own `preprocess_input`.
  The backend uses the same function, so training and serving match.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.resize_with_crop_or_pad(img, 40, 40)
    img = tf.image.random_crop(img, (32, 32, 3))
    return img, label

def to_model_input(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, (INPUT_SIZE, INPUT_SIZE), method="bilinear")
    return preprocess_input(img), label

def make_ds(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10000, seed=SEED, reshuffle_each_iteration=True).map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(to_model_input, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(x_train, y_train, True)
val_ds   = make_ds(x_val,   y_val,   False)
test_ds  = make_ds(x_test,  y_test,  False)

## 4. Build the model

The model is built as one flat network (no model-inside-a-model). This keeps the layer `conv5_block3_out` reachable, which the backend's **Grad-CAM** needs.

In [ ]:
def build_model():
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(INPUT_SIZE, INPUT_SIZE, 3))
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions")(x)
    return keras.Model(base.input, out, name="resnet50_cifar100")

def set_trainable(model, unfreeze_prefixes=()):
    """Head is always trainable. Backbone is frozen unless its layer name starts with a given prefix.
    BatchNorm layers stay frozen in both phases (standard, stable fine-tuning practice)."""
    for l in model.layers:
        if l.name in ("gap", "head_dropout", "predictions"):
            l.trainable = True
        elif isinstance(l, layers.BatchNormalization):
            l.trainable = False
        else:
            l.trainable = bool(unfreeze_prefixes) and l.name.startswith(tuple(unfreeze_prefixes))

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
    )

model = build_model()
print("Layers:", len(model.layers), "| parameters:", f"{model.count_params():,}")
print("Grad-CAM layer present:", model.get_layer("conv5_block3_out").name)

## 5. Phase 1 - train the new head only (fast)

The ImageNet knowledge stays untouched. Only the final 100-class layer learns. Expect roughly 50-65% validation accuracy here; that is normal.

In [ ]:
set_trainable(model, unfreeze_prefixes=())
compile_model(model, LR_HEAD)

t0 = time.time()
hist1 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
    callbacks=[keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "log_phase1.csv"))],
)
print(f"Phase 1 done in {(time.time() - t0) / 60:.1f} min")

## 6. Phase 2 - fine-tune the deeper blocks (this is the long step)

We unfreeze `conv4` and `conv5` blocks and train them with a small learning rate. The best epoch (highest validation accuracy) is saved automatically, and training stops early if it stops improving.

In [ ]:
set_trainable(model, unfreeze_prefixes=("conv4_", "conv5_"))
compile_model(model, LR_FINE)
print("Trainable parameters now:", f"{sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}")

BEST_PATH = os.path.join(OUT_DIR, "resnet50_cifar100_best.keras")
callbacks = [
    keras.callbacks.ModelCheckpoint(BEST_PATH, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", mode="max", factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True, verbose=1),
    keras.callbacks.CSVLogger(os.path.join(OUT_DIR, "log_phase2.csv")),
]

t0 = time.time()
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=callbacks)
print(f"Phase 2 done in {(time.time() - t0) / 60:.1f} min")

## 7. Training curves

In [ ]:
def joined(key):
    return hist1.history[key] + hist2.history[key]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(joined("accuracy"), label="train"); ax[0].plot(joined("val_accuracy"), label="validation")
ax[0].axvline(EPOCHS_HEAD - 0.5, ls="--", c="gray"); ax[0].set_title("Accuracy (dashed = fine-tuning starts)"); ax[0].legend()
ax[1].plot(joined("loss"), label="train"); ax[1].plot(joined("val_loss"), label="validation")
ax[1].axvline(EPOCHS_HEAD - 0.5, ls="--", c="gray"); ax[1].set_title("Loss"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150); plt.show()

## 8. Evaluate on the test set

The test images were never used for training or for choosing the best epoch, so this is your honest final number.

In [ ]:
best_model = keras.models.load_model(BEST_PATH)
loss, top1, top5 = best_model.evaluate(test_ds, verbose=0)
print(f"TEST  top-1 accuracy: {top1:.2%}")
print(f"TEST  top-5 accuracy: {top5:.2%}")

probs = best_model.predict(test_ds, verbose=0)
pred = probs.argmax(1)
per_class = np.array([(pred[y_test == c] == c).mean() for c in range(NUM_CLASSES)])
order = np.argsort(per_class)
print("\nWeakest 10 classes:")
for c in order[:10]:
    print(f"  {CLASS_NAMES[c]:<14} {per_class[c]:.0%}")
print("\nStrongest 5 classes:")
for c in order[::-1][:5]:
    print(f"  {CLASS_NAMES[c]:<14} {per_class[c]:.0%}")

In [ ]:
# Sample predictions: green = correct, red = wrong
fig, axes = plt.subplots(3, 8, figsize=(18, 7.5))
pick = np.random.RandomState(1).choice(len(x_test), 24, replace=False)
for ax, i in zip(axes.ravel(), pick):
    ok = pred[i] == y_test[i]
    ax.imshow(x_test[i]); ax.axis("off")
    ax.set_title(f"{CLASS_NAMES[pred[i]]}\n{probs[i].max():.0%}", fontsize=9, color="green" if ok else "red")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sample_predictions.png"), dpi=130); plt.show()

## 9. Save the files your backend needs

Copy these three into `backend/models/` and set `MODEL_BACKEND=custom` in `backend/.env`.

In [ ]:
FINAL_PATH = os.path.join(OUT_DIR, "resnet50_cifar100.keras")
best_model.save(FINAL_PATH)

with open(os.path.join(OUT_DIR, "cifar100_classes.json"), "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

config = {
    "model_file": "resnet50_cifar100.keras",
    "input_size": INPUT_SIZE,
    "preprocess": "resnet",
    "num_classes": NUM_CLASSES,
    "test_top1": round(float(top1), 4),
    "test_top5": round(float(top5), 4),
    "backend_env": {"MODEL_BACKEND": "custom", "INPUT_SIZE": INPUT_SIZE, "PREPROCESS": "resnet"},
}
with open(os.path.join(OUT_DIR, "training_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(os.listdir(OUT_DIR))
print(f"Model size: {os.path.getsize(FINAL_PATH) / 1e6:.0f} MB")

## 10. Check that the backend will behave the same

The backend loads an image with PIL, resizes it, and applies `preprocess_input`. This cell repeats exactly that and compares with the training pipeline. Agreement should be about 100%.

In [ ]:
from PIL import Image, ImageOps

def backend_style(arr_uint8):
    img = Image.fromarray(arr_uint8).convert("RGB")
    img = ImageOps.fit(img, (INPUT_SIZE, INPUT_SIZE), Image.BILINEAR)
    return preprocess_input(np.asarray(img, dtype="float32").copy())

n = 200
batch = np.stack([backend_style(x_test[i]) for i in range(n)])
p_backend = best_model.predict(batch, verbose=0).argmax(1)
print(f"Backend-style vs training-pipeline agreement: {(p_backend == pred[:n]).mean():.0%}")
print(f"Backend-style accuracy on these {n} images:  {(p_backend == y_test[:n]).mean():.0%}")

## 11. (Optional) Try your own real photos

CIFAR-100 images are tiny 32x32 pictures. Big, sharp phone or Google photos look different, so accuracy on them can be lower than the test-set number above. Try a few to see. If results are weak on real photos, that is a known limit of CIFAR-100 training data, not a bug.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        with open(name, "wb") as f:
            f.write(data)
        img = ImageOps.exif_transpose(Image.open(name)).convert("RGB")
        img = ImageOps.fit(img, (INPUT_SIZE, INPUT_SIZE), Image.BILINEAR)
        p = best_model.predict(preprocess_input(np.asarray(img, dtype="float32").copy())[None], verbose=0)[0]
        top3 = p.argsort()[::-1][:3]
        plt.figure(figsize=(3, 3.4)); plt.imshow(img); plt.axis("off")
        plt.title("\n".join(f"{CLASS_NAMES[i]} {p[i]:.0%}" for i in top3), fontsize=9); plt.show()
except ImportError:
    print("Not on Colab - copy your images in and adapt the path.")

## 12. Download the results (Colab)

In [ ]:
try:
    import shutil
    from google.colab import files
    shutil.make_archive("smartvision_model_files", "zip", OUT_DIR)
    files.download("smartvision_model_files.zip")
except ImportError:
    print("Files are in:", os.path.abspath(OUT_DIR))

## What to do next

1. Copy `resnet50_cifar100.keras`, `cifar100_classes.json` (and optionally `training_config.json`) into `backend/models/`.
2. In `backend/.env` set `MODEL_BACKEND=custom`, `INPUT_SIZE=224`, `PREPROCESS=resnet`.
3. Restart the backend and open `http://localhost:8000/docs` to test `/predict` and `/explain`.
4. Note your **test top-1 / top-5** numbers for your report.